[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/09_Manantiales_Bofedales/01_manantiales_bofedales.ipynb)

# Manantiales y bofedales: proxies topograficos e hidricos
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [1]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

Librerías listas.
earthengine-api: 1.7.39
geemap: 0.38.8


In [2]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "gen-lang-client-0636031685"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

Earth Engine inicializado correctamente.
Proyecto: gen-lang-client-0636031685


## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [3]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

Zona: Escoma, La Paz (prueba)
Ventana analítica: 1996-01-01 -> 2026-12-31 (31 años calendario)
Años completos para métricas: 1996 - 2025
Calendario: siembra mes 11 | cosecha mes 4
Umbrales activos: {'helada_c': 0.0, 'helada_grave_c': -2.0, 'mes_seco_mm': 30.0, 'spi_severo': -1.5, 'ndvi_anomalia_z': -1.5, 'manantial_occurrencia': 100, 'manantial_drenaje_max': 50, 'bofedal_altitud_min': 3000, 'bofedal_altitud_max': 5200, 'bofedal_pendiente_max': 5.0, 'bofedal_seasonality': 8, 'bofedal_ndvi_seco': 0.25, 'clases_agricolas': [12, 14]}


In [4]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

Utilidades: apilar(), reducir_bandas(), composite_mensual(), promediar_aoi(), serie_aoi()


In [5]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

Mounted at /content/drive
Descargando AOI desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N
To: /content/zona_aoi.geojson
100%|██████████| 6.32k/6.32k [00:00<00:00, 3.30MB/s]


AOI cargado desde: /content/zona_aoi.geojson
Zona: Escoma, La Paz (prueba)
Área del AOI: 5,638.9 km²
Extensión (bbox): [-68.088142, -19.88671, -67.170426, -19.050138]
Puntos de saber local cargados: {'manantiales': 0, 'bofedales': 0}


Map(center=[-19.436538457151876, -67.61413826547425], controls=(WidgetControl(options=['position', 'transparen…

# M9. Manantiales: mapa de candidatos por criterios topográficos e hídricos (proxy)
**Dimensiones:** *ocurrencia* (agua permanente en cabecera), *proximidad* (drenaje bajo),
*criterio múltiple* (priorización).

> **⚠️ Advertencia:** no existe un dataset global de manantiales confiable y gratuito.
> Este módulo produce **candidatos** (hipótesis de campo) que **deben validarse** con las
> comunidades y, si es posible, con datos de caudal o sensor de humedad.

**Criterios (todos configurables en UMBRALES de CONFIG):**
1. **Agua permanente** en JRC: ocurrencia ≥ `manantial_occurrencia` (100 = siempre agua),
2. **Cabecera de cuenca:** acumulación de flujo ≤ `manantial_drenaje_max` celdas
   (poca área aguas arriba = nacimiento, no río grande),
3. **Pendiente moderada** (≤ 25°) para descartar acantilados sin acceso.

* **Requiere ejecutar previamente:** M1 (relieve y drenaje) y M8 (ocurrencia JRC).

<details>
<summary><b>📐 Cómo priorizar los candidatos en campo</b> — clic para ver</summary>
<br>

1. Ordenar la tabla por **ocurrencia** descendente (candidatos con agua más constante).
2. Descartar los que coinciden con embalses/presas o lagunas grandes ya inventariadas.
3. Contrastar con el **saber local** (`PUNTOS_LOCALES["manantiales"]` de CONFIG): los puntos
   conocidos deben caer cerca de candidatos — si no caen, los umbrales están demasiado estrictos.
4. Visitar con **caudalímetro o datalogger de humedad** los 10–20 primeros para confirmar.

</details>

In [10]:
# ============================================================
# M1–M8 · Datos base (PREREQUISITOS de M9, M10, M8.3, M13-M15)
# ============================================================
# Sin esta celda, M9 falla con:
#   NameError: name 'ocurrencia' is not defined
# porque los módulos M1, M7 y M8 del notebook original NO existen.
#
#   M1 -> elev, pend, drenaje                  (relieve + drenaje MERIT)
#   M7 -> ndvi, anios_ndvi, ndvi_seco_map      (MODIS NDVI 250 m, 2000+)
#   M8 -> jrc, ocurrencia, estacionalidad      (JRC agua superficial 1984-2021)
#   Ut -> pix_km2                              (área de píxel km²)

# --- M1. Relieve + drenaje (MERIT Hydro) ----------------------------------
DEM  = ee.Image("USGS/SRTMGL1_003")
elev = DEM.select("elevation").rename("elev").clip(roi)
pend = ee.Terrain.slope(elev).rename("pend").clip(roi)

# MERIT Hydro: banda "upa" = área acumulada aguas arriba (km²).
# Se guarda como VALOR CONTINUO (para filtrar cabeceras en M9).
drenaje = (
    ee.Image("MERIT/Hydro/v1_0_1").select("upa")
      .clip(roi).unmask(0).rename("drenaje")
)

vis_elev = {"min": 3000, "max": 5500,
            "palette": ["#006837", "#a6d96a", "#ffffbf", "#fdae61", "#a50026"]}
vis_drenaje = {"palette": ["00a000"]}

# --- M7. NDVI MODIS 250 m (arranca 2000, no 1996) -------------------------
INICIO_NDVI = max(FECHA_INICIO, "2000-02-18")

ndvi = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
      .filterDate(INICIO_NDVI, FECHA_FIN)
      .filterBounds(roi)
      .select("NDVI")
      .map(lambda img: img.multiply(0.0001)
             .copyProperties(img, ["system:time_start"])
             .set("year", ee.Date(img.get("system:time_start")).get("year")))
)
anios_ndvi = list(range(int(INICIO_NDVI[:4]), ANIO_FIN + 1))

_meses_sc = sorted(CALENDARIO["meses_estacion_seca"])
ndvi_seco_map = (
    ndvi.filter(ee.Filter.calendarRange(_meses_sc[0], _meses_sc[-1], "month"))
        .mean().rename("ndvi_seco").clip(roi)
)

# --- M8. JRC Global Surface Water (imagen multibanda, NO colección) -------
jrc            = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
ocurrencia     = jrc.select("occurrence").clip(roi).unmask(0)
estacionalidad = jrc.select("seasonality").clip(roi).unmask(0)
recurrencia    = jrc.select("recurrence").clip(roi).unmask(0)

# --- Utilidad: área de píxel en km² ---------------------------------------
pix_km2 = ee.Image.pixelArea().divide(1e6).clip(roi)

print("Prerequisitos M1/M7/M8 listos.")
print(f"  NDVI MODIS disponible: {anios_ndvi[0]} – {anios_ndvi[-1]}")
print("  Variables: elev, pend, drenaje | ndvi, anios_ndvi, ndvi_seco_map |",
      "jrc, ocurrencia, estacionalidad, recurrencia | pix_km2")

Prerequisitos M1/M7/M8 listos.
  NDVI MODIS disponible: 2000 – 2026
  Variables: elev, pend, drenaje | ndvi, anios_ndvi, ndvi_seco_map | jrc, ocurrencia, estacionalidad, recurrencia | pix_km2


In [14]:
# ============================================================
# M9.1 Criterios, polígonos candidatos y tabla priorizada
# ============================================================

occ_min  = UMBRALES["manantial_occurrencia"]     # p. ej. 100 %
dren_max = UMBRALES["manantial_drenaje_max"]     # p. ej. 50 km² (cabeceras)

cand_mask = (
    ocurrencia.gte(occ_min)
    .And(drenaje.gt(0))              # excluir celdas sin dato
    .And(drenaje.lte(dren_max))      # poca área aguas arriba = naciente
    .And(pend.lte(25))               # pendiente moderada
).selfMask()

candidatos = cand_mask.reduceToVectors(
    reducer=ee.Reducer.countEvery(),
    geometry=roi,
    scale=ESCALAS["jrc"],
    maxPixels=1e9,
    bestEffort=True,
    eightConnected=True,
)

info_cand = (
    ee.Image.cat([ocurrencia.rename("occ"),
                  elev.rename("alt"),
                  pend.rename("pend")])
      .reduceRegions(collection=candidatos,
                     reducer=ee.Reducer.mean(),
                     scale=ESCALAS["jrc"],
                     tileScale=8)
      .map(lambda f: f.set(
          "km2",
          ee.Number(f.get("count")).multiply(ESCALAS["jrc"] ** 2).divide(1e6)))
)

n_cand = info_cand.size().getInfo()
print(f"Polígonos candidatos a manantiales: {n_cand}")

if n_cand == 0:
    print("⚠️  Sin candidatos: relaja UMBRALES['manantial_occurrencia'] "
          "(p. ej. 90) o sube 'manantial_drenaje_max' (p. ej. 100) y reejecuta.")
    df_mana = pd.DataFrame(columns=["lon","lat","occ_pct","alt_msnm",
                                    "pend_grados","km2"])
else:
    def _xy(f):
        c = f.geometry().centroid(maxError=30)
        return f.set({"lon": c.coordinates().get(0),
                      "lat": c.coordinates().get(1)})

    fc_top = (ee.FeatureCollection(info_cand.map(_xy))
              .sort("occ", ascending=False).limit(20).getInfo())

    df_mana = pd.DataFrame([{
        "lon":         round((f["properties"].get("lon")  or 0), 5),
        "lat":         round((f["properties"].get("lat")  or 0), 5),
        "occ_pct":     round((f["properties"].get("occ")  or 0), 1),
        "alt_msnm":    round((f["properties"].get("alt")  or 0), 0),
        "pend_grados": round((f["properties"].get("pend") or 0), 1),
        "km2":         round((f["properties"].get("km2")  or 0), 3),
    } for f in fc_top["features"]])

display(df_mana)
print("Criterio de orden: ocurrencia descendente. Validar en campo.")

Map_m9 = geemap.Map()
Map_m9.centerObject(roi, ZONA["zoom"])
Map_m9.add_basemap("HYBRID")
Map_m9.addLayer(elev, vis_elev, "Elevación (msnm)", False)
Map_m9.addLayer(
    info_cand.style({"color": "orange", "fillColor": "00000000", "width": 2}),
    {}, f"Candidatos a manantiales ({n_cand})")
if PUNTOS_LOCALES["manantiales"]:
    Map_m9.addLayer(ee.Geometry.MultiPoint(PUNTOS_LOCALES["manantiales"]),
                    {"color": "cyan"}, "Manantiales conocidos (saber local)")
Map_m9.addLayer(borde_aoi(), {}, "AOI")
Map_m9

Polígonos candidatos a manantiales: 0
⚠️  Sin candidatos: relaja UMBRALES['manantial_occurrencia'] (p. ej. 90) o sube 'manantial_drenaje_max' (p. ej. 100) y reejecuta.


,lon,lat,occ_pct,alt_msnm,pend_grados,km2


Criterio de orden: ocurrencia descendente. Validar en campo.


EEException: Collection.style, argument 'color': Invalid type.
Expected type: String.
Actual type: Dictionary<Object>.
Actual value: {color=orange, fillColor=00000000, width=2}

In [15]:
# ============================================================
# M10.1 Criterios combinados, áreas por clase y mapa
# ============================================================

alt_ok  = elev.gte(UMBRALES["bofedal_altitud_min"]).And(
          elev.lte(UMBRALES["bofedal_altitud_max"]))
pend_ok = pend.lte(UMBRALES["bofedal_pendiente_max"])
humedo  = estacionalidad.gte(UMBRALES["bofedal_seasonality"])
verde   = ndvi_seco_map.gte(UMBRALES["bofedal_ndvi_seco"])

base     = alt_ok.And(pend_ok)                 # planicie altoandina
cand_bof = base.And(humedo).And(verde)         # estricto

clases_bof = {
    "c_bof":        (cand_bof, "Bofedal candidato (estricto)"),
    "c_perm":       (base.And(estacionalidad.gte(10)),
                     "Con agua casi todo el año (10-12 meses)"),
    "c_estac":      (base.And(estacionalidad.gte(UMBRALES["bofedal_seasonality"]))
                         .And(estacionalidad.lt(10)),
                     "Húmedo estacional (8-9 meses)"),
    "c_verde_solo": (base.And(verde)
                         .And(estacionalidad.lt(UMBRALES["bofedal_seasonality"])),
                     "Verde en seco sin agua JRC (riego/humedad freática)"),
}

bof_imgs = [pix_km2.updateMask(masc).rename(cl)
            for cl, (masc, _) in clases_bof.items()]

d_bof = reducir_bandas(bof_imgs, list(clases_bof.keys()),
                       ESCALAS["jrc"], reductor=ee.Reducer.sum(),
                       params=REDUCIR_AREA)

df_bof = pd.DataFrame({
    "clase": [et for _, et in clases_bof.values()],
    "km2":   [d_bof.get(cl, 0) or 0 for cl in clases_bof.keys()],
})
bof_km2 = float(d_bof.get("c_bof") or 0)

print(f"Bofedales candidatos (estricto): {bof_km2:,.2f} km² "
      f"({100 * bof_km2 / area_km2:.2f}% del AOI)")
display(df_bof.round(2))

vis_bof = {"min": 0, "max": 12,
           "palette": ["ffffff","edf8fb","ccece6","99d8c9",
                       "66c2a4","2ca25f","006d2c"]}

Map_m10 = geemap.Map()
Map_m10.centerObject(roi, ZONA["zoom"])
Map_m10.add_basemap("HYBRID")
Map_m10.addLayer(ndvi_seco_map,
                 {"min": 0, "max": 0.8,
                  "palette": ["f7fcf5","c7e9c0","74c476","238b45","005a32"]},
                 "NDVI estación seca", False)
Map_m10.addLayer(estacionalidad, vis_bof,
                 "Estacionalidad JRC (meses con agua)", False)
Map_m10.addLayer(cand_bof.selfMask(), {"palette": ["1b8a3c"]},
                 "Bofedales candidatos (estricto)")
if PUNTOS_LOCALES["bofedales"]:
    Map_m10.addLayer(ee.Geometry.MultiPoint(PUNTOS_LOCALES["bofedales"]),
                     {"color": "magenta"}, "Bofedales conocidos (saber local)")
Map_m10.addLayer(borde_aoi(), {}, "AOI")
Map_m10

Bofedales candidatos (estricto): 0.00 km² (0.00% del AOI)


,clase,km2
0,Bofedal candidato (estricto),0.00
1,Con agua casi todo el año (10-12 meses),0.48
2,Húmedo estacional (8-9 meses),0.37
3,Verde en seco sin agua JRC (riego/humedad freá...,4.28


Map(center=[-19.436538457151876, -67.61413826547425], controls=(WidgetControl(options=['position', 'transparen…

In [20]:
# ============================================================
# M10.2 Clasificación supervisada - Random Forest
# ============================================================
import geemap
import ee

Map = geemap.Map()
Map.centerObject(roi, ZONA["zoom"])
Map.add_basemap('SATELLITE')

# 1) Obtener una imagen Sentinel-2 SR libre de nubes como base
def mascara_nubes(img):
    qa = img.select('MSK_CLDPRB')
    mascara = qa.lt(20) # filtrar píxeles con probabilidad de nube < 20%
    return img.updateMask(mascara)

s2_coleccion = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .map(mascara_nubes)
)

# Crear compuesto de mediana
img_base = s2_coleccion.median().clip(roi)

# 2) Calcular los índices espectrales requeridos
ndvi  = img_base.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi  = img_base.normalizedDifference(['B3', 'B8']).rename('NDWI')
mndwi = img_base.normalizedDifference(['B3', 'B11']).rename('MNDWI')
ndmi  = img_base.normalizedDifference(['B8', 'B11']).rename('NDMI')

# SAVI
savi = img_base.expression(
    '((B8 - B4) / (B8 + B4 + 0.5)) * 1.5',
    {
        'B8': img_base.select('B8'),
        'B4': img_base.select('B4')
    }
).rename('SAVI')

# AWEI
awei = img_base.expression(
    '4 * (B3 - B11) - (0.25 * B8 + 2.75 * B12)',
    {
        'B3': img_base.select('B3'),
        'B8': img_base.select('B8'),
        'B11': img_base.select('B11'),
        'B12': img_base.select('B12')
    }
).rename('AWEI')

# IBI
ibi = img_base.expression(
    '(2 * B11 / (B11 + B8) - (B8 / (B8 + B4) + B3 / (B3 + B11))) / (2 * B11 / (B11 + B8) + (B8 / (B8 + B4) + B3 / (B3 + B11)))',
    {
        'B3': img_base.select('B3'),
        'B4': img_base.select('B4'),
        'B8': img_base.select('B8'),
        'B11': img_base.select('B11')
    }
).rename('IBI')

# Construir el stack combinando las bandas reflectivas y los índices creados
stack = img_base.addBands([ndvi, ndwi, mndwi, ndmi, savi, awei, ibi])

# Seleccionar las variables predictoras
predictores = stack.select(['B2','B3','B4','B8','B11','B12',
                            'NDVI','NDWI','MNDWI','NDMI','SAVI','AWEI','IBI'])

# ------------------------------------------------------------
# Muestras de entrenamiento (ajusta con tus puntos GPS reales)
# ------------------------------------------------------------
# Bofedal: zonas húmedas verdes (ejemplo adaptado al bbox actual del AOI)
bofedal_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-67.62, -19.45]), {'clase': 1}),
    ee.Feature(ee.Geometry.Point([-67.55, -19.38]), {'clase': 1}),
])

# No bofedal: suelo seco, roca, pajonal seco
no_bofedal_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-67.90, -19.70]), {'clase': 0}),
    ee.Feature(ee.Geometry.Point([-67.25, -19.15]), {'clase': 0}),
])

muestras = bofedal_pts.merge(no_bofedal_pts)

# ------------------------------------------------------------
# Entrenamiento Random Forest
# ------------------------------------------------------------
bandas = ['B2','B3','B4','B8','B11','B12',
          'NDVI','NDWI','MNDWI','NDMI','SAVI','AWEI','IBI']

training = predictores.sampleRegions(
    collection=muestras,
    properties=['clase'],
    scale=10,
    tileScale=4
)

clasificador = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    variablesPerSplit=4,
    minLeafPopulation=2
).train(training, 'clase', bandas)

clasificacion = predictores.classify(clasificador)

# Máscara solo bofedales
bofedal_rf = clasificacion.eq(1).rename('BOFEDAL_RF')

Map.addLayer(bofedal_rf.selfMask(),
             {'palette': ['00FF00']},
             'M10.2 Bofedal (Random Forest)', True)
Map.addLayer(borde_aoi(), {}, "AOI")
Map

Map(center=[-19.436538457151894, -67.61413826547428], controls=(WidgetControl(options=['position', 'transparen…

In [23]:
# ============================================================
# M10.3 Manantiales puntuales + buffer hidrológico
# ============================================================
# Reutiliza las máscaras y capas de los módulos M9 y M10.1
manantial = cand_mask
bofedal = cand_bof

# Vectorizar manantiales (centroides)
manantiales_pts = manantial.selfMask().reduceToVectors(
    geometry=roi,
    scale=10,
    geometryType='centroid',
    reducer=ee.Reducer.countEvery(),
    maxPixels=1e13,
    bestEffort=True
)

# Área individual de cada manantial (ha)
def area_manantial(feat):
    return feat.set('area_ha',
                    ee.Number(feat.geometry().area()).divide(10000))

manantiales_pts = manantiales_pts.map(area_manantial)

print("N° manantiales detectados:", manantiales_pts.size().getInfo())

# ------------------------------------------------------------
# Buffer de influencia (zona de recarga / descarga)
# ------------------------------------------------------------
buffer_100 = manantiales_pts.map(lambda f: f.buffer(100))
buffer_250 = manantiales_pts.map(lambda f: f.buffer(250))
buffer_500 = manantiales_pts.map(lambda f: f.buffer(500))

Map.addLayer(manantiales_pts, {'color': 'cyan'}, 'M10.3 Manantiales (pts)')
Map.addLayer(buffer_100, {'color': 'yellow'}, 'Buffer 100 m', False)
Map.addLayer(buffer_250, {'color': 'orange'}, 'Buffer 250 m', False)
Map.addLayer(buffer_500, {'color': 'red'},    'Buffer 500 m', False)

# ------------------------------------------------------------
# Bofedales asociados a manantiales (dentro del buffer 250 m)
# ------------------------------------------------------------
# Convertimos el buffer a una máscara rasterizada para filtrar la imagen bofedal
mask_buffer = ee.Image.constant(1).clip(buffer_250).mask()
bofedal_cerca = bofedal.updateMask(mask_buffer)

Map.addLayer(bofedal_cerca.selfMask(),
             {'palette': ['00AA00']},
             'M10.3 Bofedal cercano a manantial', True)

Map

N° manantiales detectados: 0


Map(bottom=73095.0, center=[-19.54684919724742, -67.36816406250001], controls=(WidgetControl(options=['positio…

In [25]:
# ============================================================
# M10.4 Distribución altitudinal de bofedales y manantiales
# ============================================================
srtm = ee.Image('USGS/SRTMGL1_003').clip(roi)
elev = srtm.rename('elev')

# Pisos altitudinales típicos en Puno
pisos = ee.Image(0).where(elev.gte(3800).And(elev.lt(4000)), 1) \
                    .where(elev.gte(4000).And(elev.lt(4200)), 2) \
                    .where(elev.gte(4200).And(elev.lt(4400)), 3) \
                    .where(elev.gte(4400).And(elev.lt(4600)), 4) \
                    .where(elev.gte(4600).And(elev.lt(4800)), 5) \
                    .where(elev.gte(4800), 6) \
                    .rename('piso')

# Área de bofedal por piso
area_bofedal_piso = (bofedal.multiply(ee.Image.pixelArea())
                     .divide(10000)
                     .addBands(pisos)
                     .reduceRegion(
                         reducer=ee.Reducer.sum().group(1, 'piso'),
                         geometry=roi,
                         scale=10,
                         maxPixels=1e13
                     ).getInfo())

print("Bofedal por piso altitudinal:")
for g in area_bofedal_piso['groups']:
    print(f"  Piso {g['piso']}: {g['sum']:.2f} ha")

# Rango típico de bofedales en Puno
bofedal_altitud = bofedal.updateMask(elev.gte(3800).And(elev.lte(4800)))

Map.addLayer(elev, {'min':3500,'max':5500,
             'palette':['green','yellow','orange','brown','white']},
             'DEM', False)
Map.addLayer(bofedal_altitud.selfMask(),
             {'palette': ['00FF00']},
             'M10.4 Bofedal en rango 3800–4800 m', True)

Map

KeyboardInterrupt: 

In [ ]:
# ============================================================
# M10.5 Tendencia multitemporal de bofedales (2015–2025)
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

years = list(range(2015, 2026))
area_serie = []

for y in years:
    s2_y = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi)
            .filterDate(f'{y}-05-01', f'{y}-10-31')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25)))

    n_img = s2_y.size().getInfo()
    if n_img == 0:
        area_serie.append({'year': y, 'area_ha': None, 'n_img': 0})
        continue

    med = s2_y.median().divide(10000).clip(roi)
    ndvi_y = med.normalizedDifference(['B8','B4']).rename('NDVI')
    ndmi_y = med.normalizedDifference(['B8','B11']).rename('NDMI')
    mndwi_y = med.normalizedDifference(['B3','B11']).rename('MNDWI')

    bof_y = ndvi_y.gt(0.25).And(ndmi_y.gt(0.15)).And(mndwi_y.gt(-0.1))

    area = (bof_y.selfMask()
            .multiply(ee.Image.pixelArea())
            .divide(10000)
            .reduceRegion(ee.Reducer.sum(), roi, 10, maxPixels=1e13)
            .get('NDVI'))

    area_val = area.getInfo() if area else 0
    area_serie.append({'year': y, 'area_ha': area_val, 'n_img': n_img})
    print(f"{y}: {area_val:.1f} ha  ({n_img} imágenes)")

df = pd.DataFrame(area_serie).dropna()

# Gráfico
plt.figure(figsize=(9, 4.5))
plt.plot(df['year'], df['area_ha'], marker='o', color='green', lw=2)
plt.fill_between(df['year'], df['area_ha'], alpha=0.2, color='green')
plt.xlabel('Año')
plt.ylabel('Área de bofedal (ha)')
plt.title('Tendencia de bofedales en la cuenca (2015–2025)')
plt.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('tendencia_bofedales.png', dpi=150)
plt.show()

# Tasa de cambio (ha/año)
if len(df) > 1:
    tasa = (df['area_ha'].iloc[-1] - df['area_ha'].iloc[0]) / (df['year'].iloc[-1] - df['year'].iloc[0])
    print(f"\nTasa de cambio: {tasa:.2f} ha/año")

2016: 35.0 ha  (23 imágenes)
2017: 44.8 ha  (36 imágenes)
2018: 296.6 ha  (30 imágenes)


In [ ]:
# ============================================================
# M10.6 Bofedales y manantiales por microcuenca
# ============================================================
# HidroSHEDS o HydroBASINS (nivel 8/9)
cuencas = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_8').filterBounds(roi)
print("Microcuencas intersectando ROI:", cuencas.size().getInfo())

# Área de bofedal por microcuenca
def stats_micro(feat):
    geom = feat.geometry()
    bof = (bofedal.multiply(ee.Image.pixelArea()).divide(10000)
           .reduceRegion(ee.Reducer.sum(), geom, 10, maxPixels=1e13)
           .get('BOFEDAL'))
    man = (manantial.multiply(ee.Image.pixelArea()).divide(10000)
           .reduceRegion(ee.Reducer.sum(), geom, 10, maxPixels=1e13)
           .get('MANANTIAL'))
    return feat.set({
        'area_bofedal_ha': ee.Number(bof).format('%.2f'),
        'area_manantial_ha': ee.Number(man).format('%.2f')
    })

cuencas_stats = cuencas.map(stats_micro).filter(
    ee.Filter.gt('area_bofedal_ha', 0)
)

# Tabla resultante
print(cuencas_stats.limit(10).getInfo())

# Estilo
estilo_bof = {'color': '1a9850', 'width': 1.5, 'fillColor': 'a6d96a00'}
Map.addLayer(cuencas_stats.style(**estilo_bof), {}, 'M10.6 Microcuencas')

Map

In [ ]:
# ============================================================
# M10.7 Priorización de bofedales para conservación
# ============================================================
# Criterios:
#   1. Mayor área de bofedal
#   2. Cercanía a manantiales (≤ 250 m)
#   3. Mayor precipitación (CHIRPS)
#   4. Menor pendiente (más frágil)

# Precipitación media anual (CHIRPS 2024)
chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterDate('2024-01-01','2024-12-31')
          .select('precipitation')
          .sum()
          .clip(roi))

# Pendiente
srtm = ee.Image('USGS/SRTMGL1_003')
pendiente = ee.Terrain.slope(srtm).clip(roi)

# Distancia a manantiales
dist_manantial = manantial.selfMask().fastDistanceTransform(1000).sqrt() \
                  .multiply(ee.Image.pixelArea().sqrt()) \
                  .rename('dist_man')

# Normalizar variables 0–1
def norm(img):
    return img.subtract(img.reduceRegion(ee.Reducer.min(), roi, 90, maxPixels=1e13).values().get(0)) \
              .divide(
                  img.reduceRegion(ee.Reducer.max(), roi, 90, maxPixels=1e13).values().get(0)
                     .subtract(img.reduceRegion(ee.Reducer.min(), roi, 90, maxPixels=1e13).values().get(0))
              )

bofedal_norm   = bofedal.multiply(1.0)
chirps_norm    = norm(chirps)
pendiente_inv  = ee.Image(1).subtract(norm(pendiente))
dist_inv       = ee.Image(1).subtract(norm(dist_manantial))

# Índice de prioridad (pesos ajustables)
prioridad = (bofedal_norm.multiply(0.40)
             .add(chirps_norm.multiply(0.25))
             .add(pendiente_inv.multiply(0.15))
             .add(dist_inv.multiply(0.20))
             ).updateMask(bofedal).rename('PRIORIDAD')

Map.addLayer(prioridad,
             {'min': 0, 'max': 1,
              'palette': ['red','orange','yellow','lightgreen','green']},
             'M10.7 Prioridad de conservación', True)

# Guardar
geemap.ee_export_image(prioridad, filename='prioridad_bofedales.tif',
                       scale=10, region=roi, file_per_band=False)
print("✅ Exportado: prioridad_bofedales.tif")

Map

In [ ]:
# ============================================================
# M10.9 Clorofila-a y proliferación algal en manantiales/bofedales
# ============================================================
# Reutiliza: img (mediana Sentinel-2), roi, manantial, bofedal

B2 = img.select('B2')
B3 = img.select('B3')
B4 = img.select('B4')
B5 = img.select('B5')   # red-edge 1 (705 nm)
B6 = img.select('B6')   # red-edge 2 (740 nm)
B8 = img.select('B8')
B11 = img.select('B11')

# --- NDCI (Normalized Difference Chlorophyll Index) ---
ndci = B5.subtract(B4).divide(B5.add(B4)).rename('NDCI')

# --- Chl-a 2-bandas (B5/B4) ---
chla = B5.divide(B4).rename('CHLA_2B')

# --- Cyanobacteria Index (CI) ---
ci = B4.subtract(B5).divide(B4).multiply(B4.divide(B2)).rename('CI')

# --- FAI (Floating Algae Index) ---
lambda_r = 665; lambda_nir = 842; lambda_swir = 1610
fai = B8.subtract(
    B4.add(B11.subtract(B4)
           .multiply(lambda_nir - lambda_r)
           .divide(lambda_swir - lambda_r))
).rename('FAI')

# Máscara solo sobre superficies de agua (manantiales + bofedales húmedos)
mask_agua = manantial.Or(bofedal)

ndci_agua = ndci.updateMask(mask_agua)
chla_agua = chla.updateMask(mask_agua)
ci_agua   = ci.updateMask(mask_agua)
fai_agua  = fai.updateMask(mask_agua)

# ------------------------------------------------------------
# Clasificación de nivel trófico (umbrales NDCI típicos)
# ------------------------------------------------------------
# NDCI < 0.05   → oligotrófico (agua limpia)
# 0.05–0.15     → mesotrófico
# 0.15–0.30     → eutrófico
# > 0.30        → hipereutrófico (floración)

nivel_trofico = (ee.Image(0)
    .where(ndci_agua.gte(0.05).And(ndci_agua.lt(0.15)), 1)
    .where(ndci_agua.gte(0.15).And(ndci_agua.lt(0.30)), 2)
    .where(ndci_agua.gte(0.30), 3)
    .updateMask(mask_agua)
    .rename('NIVEL_TROFICO'))

Map.addLayer(ndci_agua,
             {'min': -0.1, 'max': 0.4,
              'palette': ['blue','cyan','yellow','orange','red']},
             'M10.9 NDCI – Clorofila-a', True)
Map.addLayer(chla_agua,
             {'min': 0.8, 'max': 2.0,
              'palette': ['white','lightgreen','green','darkgreen']},
             'M10.9 Chl-a (B5/B4)', False)
Map.addLayer(ci_agua,
             {'min': -0.05, 'max': 0.10,
              'palette': ['white','yellow','red']},
             'M10.9 CI – Cianobacterias', False)
Map.addLayer(fai_agua,
             {'min': -0.01, 'max': 0.02,
              'palette': ['white','lightgreen','green']},
             'M10.9 FAI – Algas flotantes', False)
Map.addLayer(nivel_trofico,
             {'min': 0, 'max': 3,
              'palette': ['blue','green','yellow','red']},
             'M10.9 Nivel trófico', False)

Map

In [ ]:
# ============================================================
# M10.10 Turbidez y sólidos suspendidos en manantiales
# ============================================================
# --- NDTI (Normalized Difference Turbidity Index) ---
ndti = B4.subtract(B3).divide(B4.add(B3)).rename('NDTI')

# --- Ratio rojo/verde como proxy de turbidez ---
turb_ratio = B4.divide(B3).rename('TURB_RATIO')

# --- Índice de claridad (blue/red) ---
claridad = B2.divide(B4).rename('CLARIDAD')

# --- TSS proxy (Neckles et al. adaptado) ---
tss = B4.multiply(1148.4).add(111.2).rename('TSS_mgL')

# Máscara solo agua
ndti_agua    = ndti.updateMask(manantial)
turb_agua    = turb_ratio.updateMask(manantial)
clar_agua    = claridad.updateMask(manantial)
tss_agua     = tss.updateMask(manantial)

# ------------------------------------------------------------
# Clasificación de turbidez
# ------------------------------------------------------------
# NDTI < 0     → agua clara
# 0–0.10       → ligera
# 0.10–0.20    → moderada
# > 0.20       → alta (posible contaminación)

clase_turb = (ee.Image(0)
    .where(ndti_agua.gte(0.0).And(ndti_agua.lt(0.10)), 1)
    .where(ndti_agua.gte(0.10).And(ndti_agua.lt(0.20)), 2)
    .where(ndti_agua.gte(0.20), 3)
    .updateMask(manantial)
    .rename('TURBIDEZ'))

Map.addLayer(ndti_agua,
             {'min': -0.1, 'max': 0.3,
              'palette': ['blue','cyan','yellow','brown']},
             'M10.10 NDTI – Turbidez', True)
Map.addLayer(turb_agua,
             {'min': 0.8, 'max': 1.6,
              'palette': ['blue','white','brown']},
             'M10.10 Ratio B4/B3', False)
Map.addLayer(clar_agua,
             {'min': 0.5, 'max': 1.5,
              'palette': ['brown','white','blue']},
             'M10.10 Claridad (B2/B4)', False)
Map.addLayer(tss_agua,
             {'min': 0, 'max': 100,
              'palette': ['blue','lightblue','yellow','orange','brown']},
             'M10.10 TSS (mg/L aprox)', False)
Map.addLayer(clase_turb,
             {'min': 0, 'max': 3,
              'palette': ['blue','cyan','yellow','brown']},
             'M10.10 Clase de turbidez', False)

Map

In [ ]:
# ============================================================
# M10.11 Índice de Calidad de Agua (ICA) en manantiales
# ============================================================
# Criterios (todos normalizados 0–1, mayor = mejor calidad):
#   - NDCI bajo  → poca clorofila  → buena
#   - NDTI bajo  → poca turbidez   → buena
#   - NDWI alto  → cuerpo de agua  → activo
#   - MNDWI alto → agua clara      → buena

def norm01(img, roi, scale=90):
    stats = img.reduceRegion(ee.Reducer.minMax(), roi, scale, maxPixels=1e13)
    mn = ee.Number(stats.get(img.bandNames().get(0).cat('_min')))
    mx = ee.Number(stats.get(img.bandNames().get(0).cat('_max')))
    return img.subtract(mn).divide(mx.subtract(mn))

ndci_norm = norm01(ndci, roi)
ndti_norm = norm01(ndti, roi)
ndwi_norm = norm01(ndwi, roi)
mndwi_norm = norm01(mndwi, roi)

# Invertir clorofila y turbidez (menor = mejor)
ica = (ee.Image(1).subtract(ndci_norm).multiply(0.30)
       .add(ee.Image(1).subtract(ndti_norm).multiply(0.30))
       .add(ndwi_norm.multiply(0.20))
       .add(mndwi_norm.multiply(0.20))
       ).updateMask(manantial).rename('ICA')

# Clasificación
ica_clase = (ee.Image(0)
    .where(ica.gte(0.25), 1)   # regular
    .where(ica.gte(0.50), 2)   # buena
    .where(ica.gte(0.75), 3)   # excelente
    .updateMask(manantial)
    .rename('ICA_CLASE'))

Map.addLayer(ica,
             {'min': 0, 'max': 1,
              'palette': ['red','orange','yellow','lightgreen','green']},
             'M10.11 ICA – Calidad de agua', True)
Map.addLayer(ica_clase,
             {'min': 0, 'max': 3,
              'palette': ['red','orange','lightgreen','green']},
             'M10.11 Clase ICA', False)

# Estadísticas
ica_stats = ica.reduceRegion(
    ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    roi, 10, maxPixels=1e13).getInfo()
print("ICA en manantiales:", ica_stats)

Map

In [ ]:
# ============================================================
# M10.12 Evolución de clorofila y turbidez en manantiales
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

years = list(range(2020, 2026))
rows = []

for y in years:
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(roi)
           .filterDate(f'{y}-05-01', f'{y}-10-31')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25)))
    if col.size().getInfo() == 0:
        continue
    med = col.median().divide(10000).clip(roi)
    b3, b4, b5 = med.select('B3'), med.select('B4'), med.select('B5')
    ndci_y = b5.subtract(b4).divide(b5.add(b4))
    ndti_y = b4.subtract(b3).divide(b4.add(b3))

    mask_y = manantial  # máscara base

    ndci_m = ndci_y.updateMask(mask_y).reduceRegion(
        ee.Reducer.mean(), roi, 10, maxPixels=1e13).get('B5')
    ndti_m = ndti_y.updateMask(mask_y).reduceRegion(
        ee.Reducer.mean(), roi, 10, maxPixels=1e13).get('B4')

    rows.append({
        'year': y,
        'NDCI': ndci_m.getInfo() if ndci_m else None,
        'NDTI': ndti_m.getInfo() if ndti_m else None
    })

df = pd.DataFrame(rows)
print(df)

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(df['year'], df['NDCI'], 'o-', color='green', label='NDCI (Chl-a)')
ax1.set_xlabel('Año'); ax1.set_ylabel('NDCI', color='green')
ax1.tick_params(axis='y', labelcolor='green')

ax2 = ax1.twinx()
ax2.plot(df['year'], df['NDTI'], 's--', color='brown', label='NDTI (Turbidez)')
ax2.set_ylabel('NDTI', color='brown')
ax2.tick_params(axis='y', labelcolor='brown')

plt.title('Evolución de calidad de agua en manantiales (2020–2025)')
fig.tight_layout()
plt.savefig('calidad_agua_manantiales.png', dpi=150)
plt.show()

## Interpretación de M10
* **Candidato estricto** es un filtro conservador: si da 0 km², relajar umbrales (p. ej.
  `bofedal_seasonality` 8→6 o `bofedal_ndvi_seco` 0,25→0,20) y reejecutar.
* **"Verde en seco sin agua JRC":** vegetación que sobrevive a la seca sin agua superficial
  detectable → señala **riego, humedad freática o manantiales** (enlaza con M9).
* **Serie NDVI de seca (M10.2):** años por debajo de la media = deterioro temporal del bofedal;
  tendencia negativa sostenida = alerta de sobreuso o cambio climático; contrastar con caudal
  medido y con el uso que reportan las comunidades.

**Preguntas guía**
1. ¿Sus bofedales conocidos caen dentro de los candidatos? ¿faltan o sobran?
2. ¿El año más débil de la serie coincide con sequías recordadas o con mayor presión de pastoreo?
3. ¿Qué partes "verdes en seco" no son bofedales sino riego agrícola (M11)?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
